# # Expernetic Data Engineer Internship Assignment
# ## Notebook 02 - Data Profiling & Data Quality Assessment
#
# Objective:
# Assess dataset quality by evaluating:
# - Missing values
# - Duplicate records
# - Primary key integrity
# - Data type issues
# - Price data validity
# - Outlier detection

# ## 1. Dataset Loading
#
# Load London and New York Airbnb listing datasets
# for data quality assessment.

In [1]:
import pandas as pd
import numpy as np

In [2]:
london_listings = pd.read_csv("../data/raw/london/listings.csv.gz")

ny_listings = pd.read_csv("../data/raw/new_york/listings.csv.gz")

In [3]:
print("London:", london_listings.shape)

print("New York:", ny_listings.shape)

London: (96871, 79)
New York: (35036, 90)


## 2. Missing Value Analysis# Objective:
# Identify columns with significant missing values
# and assess their impact on downstream analytics.

In [4]:
def missing_value_report(df):

    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isnull().sum(),
        "missing_pct": round(df.isnull().mean() * 100, 2)
    })

    return report.sort_values(
        by="missing_pct",
        ascending=False
    )

In [5]:
london_missing = missing_value_report(
    london_listings
)

ny_missing = missing_value_report(
    ny_listings
)

In [6]:
london_missing.head(20)

,column,missing_count,missing_pct
license,license,96871,100.00
calendar_updated,calendar_updated,96871,100.00
neighbourhood_group_cleansed,neighbourhood_group_cleansed,96871,100.00
neighbourhood,neighbourhood,55662,57.46
neighborhood_overview,neighborhood_overview,55663,57.46
host_neighbourhood,host_neighbourhood,51021,52.67
host_about,host_about,47038,48.56
beds,beds,34920,36.05
price,price,34908,36.04
estimated_revenue_l365d,estimated_revenue_l365d,34908,36.04


In [7]:
ny_missing.head(20)

,column,missing_count,missing_pct
neighborhood_overview,neighborhood_overview,35036,100.00
host_since,host_since,35036,100.00
host_response_time,host_response_time,35036,100.00
host_thumbnail_url,host_thumbnail_url,35036,100.00
host_acceptance_rate,host_acceptance_rate,35036,100.00
host_response_rate,host_response_rate,35036,100.00
host_verifications,host_verifications,35036,100.00
neighbourhood,neighbourhood,35036,100.00
host_total_listings_count,host_total_listings_count,35036,100.00
host_neighbourhood,host_neighbourhood,35036,100.00


## 3. Duplicate Record Analysis
# Objective:
# Verify the absence of duplicate records and
# ensure dataset uniqueness.

In [10]:
print("London Duplicate Rows:",
      london_listings.duplicated().sum())

print("NY Duplicate Rows:",
      ny_listings.duplicated().sum())

London Duplicate Rows: 0
NY Duplicate Rows: 0


In [12]:
print("London Duplicate Listing IDs:",
      london_listings["id"].duplicated().sum())

print("NY Duplicate Listing IDs:",
      ny_listings["id"].duplicated().sum())

London Duplicate Listing IDs: 0
NY Duplicate Listing IDs: 0


In [13]:
print("London Price Sample:")
print(london_listings["price"].head(10))

print("\nNY Price Sample:")
print(ny_listings["price"].head(10))

London Price Sample:
0     $70.00
1    $149.00
2    $411.00
3        NaN
4    $210.00
5    $280.00
6     $90.00
7     $61.00
8    $340.00
9     $49.00
Name: price, dtype: str

NY Price Sample:
0    $117.27
1     $66.87
2     $77.17
3    $202.47
4    $365.97
5    $193.03
6    $104.35
7     $60.84
8        NaN
9        NaN
Name: price, dtype: str


## 4. Primary Key Validation
#
# Objective:
# Validate that listing identifiers are unique
# and suitable for relationship mapping.

In [14]:
print("London Price Nulls:",
      london_listings["price"].isnull().sum())

print("NY Price Nulls:",
      ny_listings["price"].isnull().sum())

London Price Nulls: 34908
NY Price Nulls: 14343


In [15]:
london_listings["price_clean"] = (
    london_listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [16]:
print(london_listings[["price", "price_clean"]].head(10))
print(ny_listings[["price", "price_clean"]].head(10))

print("\nLondon price_clean dtype:", london_listings["price_clean"].dtype)
print("NY price_clean dtype:", ny_listings["price_clean"].dtype)

     price  price_clean
0   $70.00         70.0
1  $149.00        149.0
2  $411.00        411.0
3      NaN          NaN
4  $210.00        210.0
5  $280.00        280.0
6   $90.00         90.0
7   $61.00         61.0
8  $340.00        340.0
9   $49.00         49.0


KeyError: "['price_clean'] not in index"

In [17]:
ny_listings["price_clean"] = (
    ny_listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
print(london_listings[["price", "price_clean"]].head(10))
print(ny_listings[["price", "price_clean"]].head(10))

print("\nLondon price_clean dtype:", london_listings["price_clean"].dtype)
print("NY price_clean dtype:", ny_listings["price_clean"].dtype)

     price  price_clean
0   $70.00         70.0
1  $149.00        149.0
2  $411.00        411.0
3      NaN          NaN
4  $210.00        210.0
5  $280.00        280.0
6   $90.00         90.0
7   $61.00         61.0
8  $340.00        340.0
9   $49.00         49.0
     price  price_clean
0  $117.27       117.27
1   $66.87        66.87
2   $77.17        77.17
3  $202.47       202.47
4  $365.97       365.97
5  $193.03       193.03
6  $104.35       104.35
7   $60.84        60.84
8      NaN          NaN
9      NaN          NaN

London price_clean dtype: float64
NY price_clean dtype: float64


In [19]:
print("London zero/negative prices:",
      (london_listings["price_clean"] <= 0).sum())

print("NY zero/negative prices:",
      (ny_listings["price_clean"] <= 0).sum())

London zero/negative prices: 0
NY zero/negative prices: 0


In [20]:
print("London Price Summary")
print(london_listings["price_clean"].describe())

print("\nNY Price Summary")
print(ny_listings["price_clean"].describe())

London Price Summary
count    6.196300e+04
mean     2.299170e+02
std      4.437589e+03
min      7.000000e+00
25%      7.700000e+01
50%      1.350000e+02
75%      2.210000e+02
max      1.085147e+06
Name: price_clean, dtype: float64

NY Price Summary
count    20693.000000
mean       254.998857
std        440.478328
min          4.470000
25%         93.800000
50%        167.250000
75%        279.500000
max      15075.000000
Name: price_clean, dtype: float64


In [21]:
london_listings.nlargest(
    10,
    "price_clean"
)[
    [
        "id",
        "name",
        "room_type",
        "property_type",
        "price_clean"
    ]
]

,id,name,room_type,property_type,price_clean
83405,1389784758661918909,Lux 2 Bed in Canary Wharf close to Excel & O2,Entire home/apt,Entire rental unit,1085147.0
10228,13841484,Bright & airy DoubleBed with EnSuite in Zone 2!,Entire home/apt,Entire rental unit,74100.0
77023,1311151886101957046,Walk To London Eye,Private room,Private room in rental unit,66189.0
54442,957005187369596707,Close To London Eye,Private room,Private room in rental unit,65000.0
57204,1009450636308513568,Close to London Eye (BOL),Private room,Private room in rental unit,58000.0
75312,1289793379859667497,Very Central Room - Walk to Eye,Private room,Private room in rental unit,58000.0
9489,13254774,No Longer Available,Private room,Private room in rental unit,53588.0
38591,558514003024408716,SHORT WALK TO LONDON EYE - DOUBLE ROOM (SUR),Private room,Private room in condo,50000.0
51162,904417963453083510,Twin Room Close to London Eye (RHI),Private room,Private room in rental unit,50000.0
63728,1124036005279942799,"Spacious, airy penthouse with stunning view",Entire home/apt,Entire rental unit,30812.0


In [22]:
ny_listings.nlargest(
    10,
    "price_clean"
)[
    [
        "id",
        "name",
        "room_type",
        "property_type",
        "price_clean"
    ]
]

,id,name,room_type,property_type,price_clean
27603,1111666023012679487,Opulent Midtown Retreat with White-Gloved Service,Hotel room,Room in hotel,15075.00
27604,1111666150326773564,"Presidential Suite Park View King Bed, The Pierre",Hotel room,Room in hotel,15075.00
790,990529,Sunny Luxury Bedroom Guest Bathroom,Private room,Private room in rental unit,12279.07
914,1623431,Enormous Chrysler-View Bedroom/Bath,Private room,Private room in rental unit,12078.47
23189,830682282357157632,North Gallery Café,Entire home/apt,Tower,11061.80
23355,830656153550799267,"Eighth House, State of the Art NYC Venue",Entire home/apt,Tower,11061.80
16983,49920227,WELCOME HOME 15 MINUTES TO MANHATTAN BOOK TODAY,Private room,Private room in home,10944.83
13786,39553889,"Paper Factory, Executive King",Private room,Room in hotel,10000.00
13787,39554366,"Paper Factory, Superior Room w/ Double Beds",Private room,Room in hotel,10000.00
13788,39554839,"Paper Factory, Deluxe King",Private room,Room in hotel,10000.00


In [23]:
london_listings["price_clean"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25      77.0
0.50     135.0
0.75     221.0
0.90     360.0
0.95     500.0
0.99    1100.0
Name: price_clean, dtype: float64

In [ ]:
ny_listings["price_clean"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25      93.8000
0.50     167.2500
0.75     279.5000
0.90     476.5000
0.95     676.9140
0.99    1562.3508
Name: price_clean, dtype: float64

In [27]:
print("London Listing  > £1100:", 
      (london_listings["price_clean"] > 1100).sum())

print("NY Listing  > $1562:", 
      (ny_listings["price_clean"] > 1562).sum())

London Listing  > £1100: 613
NY Listing  > $1562: 207


## Outlier Flagging

In [ ]:
london_listings["price_outlier"] = (
    london_listings["price_clean"] > 1100
)

ny_listings["price_outlier"] = (
    ny_listings["price_clean"] > 1562
)